# NCAT AutoDrive Challenge -- CAN Log Perception Analysis

**Team:** North Carolina A&T State University  
**Date:** 2026-03-29  
**Data Source:** NeoVI CAN logger (HSCAN3 scoring bus)  
**DBC Database:** ADC2_SC_2024.2.dbc (Scoring CAN)

---

## 1. Introduction

This notebook analyzes the CAN bus data recorded during the AutoDrive Challenge test run using the team's NeoVI hardware. The data was captured on the HSCAN3 (scoring) CAN bus, which carries the autonomous driving controller's perception outputs including:

- **Autonomy state** (AV engagement, control authority)
- **Object detection** (vehicles, pedestrians, cyclists detected by camera/lidar)
- **Traffic signal recognition** (signal head position, illuminated light states)
- **Traffic sign recognition** (sign type, position, confidence)
- **Vehicle GPS location** (latitude/longitude)

The raw `.vsb` binary log was converted to CSV, then decoded using the ADC2 scoring CAN database (`ADC2_SC_2024.2.dbc`) to extract physical signal values.

## 2. Data Loading and Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta
import cantools
import csv
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

ICS_EPOCH_UNIX = 1167609600.0

DBC_PATH = os.path.join('dbc', 'ADC2_SC_2024.2.dbc')
db = cantools.database.load_file(DBC_PATH)
dbc_ids = {msg.frame_id: msg for msg in db.messages}

raw_df = pd.read_csv('NCAT_CAN_Logs.csv')
raw_df['unix_ts'] = raw_df['Timestamp_Sec'].astype(float) + ICS_EPOCH_UNIX
raw_df['time'] = pd.to_datetime(raw_df['unix_ts'], unit='s', utc=True)
t0 = raw_df['unix_ts'].iloc[0]
raw_df['elapsed_s'] = raw_df['unix_ts'] - t0

print(f'Total messages: {len(raw_df)}')
print(f'Time span: {raw_df["DateTime_UTC"].iloc[0]} to {raw_df["DateTime_UTC"].iloc[-1]}')
print(f'Duration: {raw_df["elapsed_s"].iloc[-1]:.1f} seconds')
print(f'Unique ArbIDs: {raw_df["ArbID_Dec"].nunique()}')
print(f'\nMessage counts:')
for aid, cnt in raw_df.groupby('ArbID_Dec').size().items():
    name = dbc_ids[aid].name if aid in dbc_ids else '?'
    print(f'  0x{aid:03X} ({name}): {cnt}')

Total messages: 27510
Time span: 2026-03-29 18:59:06.895005 to 2026-03-29 19:02:23.293903
Duration: 196.4 seconds
Unique ArbIDs: 12

Message counts:
  0x010 (AVLight): 1965
  0x011 (AVState): 1965
  0x014 (VehicleLocation): 1965
  0x020 (Objects): 1965
  0x021 (Object_TrackA): 1965
  0x022 (Object_TrackB): 1965
  0x023 (Object_TrackC): 1965
  0x024 (Object_TrackD): 1965
  0x030 (TrafficSignalHeads): 3930
  0x031 (TrafficSignalHead_TrackA): 3930
  0x040 (TrafficSigns): 1965
  0x041 (TrafficSign_TrackA): 1965


In [2]:
def decode_messages(df, arb_id):
    """Decode all messages for a given ArbID using the DBC."""
    msg_def = dbc_ids[arb_id]
    subset = df[df['ArbID_Dec'] == arb_id].copy()
    records = []
    for _, row in subset.iterrows():
        dlc = int(row['DLC'])
        data = bytearray()
        for i in range(dlc):
            b = row.get(f'B{i}', '')
            if pd.notna(b) and b != '':
                data.append(int(str(b), 16))
            else:
                data.append(0)
        try:
            decoded = msg_def.decode(bytes(data), decode_choices=False)
            decoded['elapsed_s'] = row['elapsed_s']
            decoded['time'] = row['time']
            records.append(decoded)
        except Exception:
            pass
    return pd.DataFrame(records)

av_light = decode_messages(raw_df, 16)
av_state = decode_messages(raw_df, 17)
vehicle_loc = decode_messages(raw_df, 20)
objects_hdr = decode_messages(raw_df, 32)
obj_track_a = decode_messages(raw_df, 33)
obj_track_b = decode_messages(raw_df, 34)
obj_track_c = decode_messages(raw_df, 35)
obj_track_d = decode_messages(raw_df, 36)
sig_hdr = decode_messages(raw_df, 48)
sig_track_a = decode_messages(raw_df, 49)
sign_hdr = decode_messages(raw_df, 64)
sign_track_a = decode_messages(raw_df, 65)

print(f'Decoded all 12 message types successfully')
print(f'  AVLight:         {len(av_light)} msgs')
print(f'  AVState:         {len(av_state)} msgs')
print(f'  VehicleLocation: {len(vehicle_loc)} msgs')
print(f'  Objects header:  {len(objects_hdr)} msgs')
print(f'  Object TrackA:   {len(obj_track_a)} msgs')
print(f'  Signal headers:  {len(sig_hdr)} msgs')
print(f'  Signal TrackA:   {len(sig_track_a)} msgs')
print(f'  Sign headers:    {len(sign_hdr)} msgs')
print(f'  Sign TrackA:     {len(sign_track_a)} msgs')

Decoded all 12 message types successfully
  AVLight:         1965 msgs
  AVState:         1965 msgs
  VehicleLocation: 1965 msgs
  Objects header:  1965 msgs
  Object TrackA:   1965 msgs
  Signal headers:  3930 msgs
  Signal TrackA:   3930 msgs
  Sign headers:    1965 msgs
  Sign TrackA:     1965 msgs


## 3. GPS Track and Vehicle Path

The VehicleLocation message (0x014) provides latitude and longitude at 10 Hz. This shows the route driven during the test.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sc = ax.scatter(vehicle_loc['VehicleLongitude'], vehicle_loc['VehicleLatitude'],
                c=vehicle_loc['elapsed_s'], cmap='viridis', s=3, zorder=2)
ax.plot(vehicle_loc['VehicleLongitude'].iloc[0], vehicle_loc['VehicleLatitude'].iloc[0],
        'go', markersize=10, label='Start', zorder=3)
ax.plot(vehicle_loc['VehicleLongitude'].iloc[-1], vehicle_loc['VehicleLatitude'].iloc[-1],
        'rs', markersize=10, label='End', zorder=3)
ax.set_xlabel('Longitude (deg)')
ax.set_ylabel('Latitude (deg)')
ax.set_title('Vehicle GPS Track')
ax.legend()
cb = plt.colorbar(sc, ax=ax)
cb.set_label('Elapsed Time (s)')
ax.set_aspect('equal')

ax = axes[1]
ax.plot(vehicle_loc['elapsed_s'], vehicle_loc['VehicleLatitude'], label='Latitude')
ax2 = ax.twinx()
ax2.plot(vehicle_loc['elapsed_s'], vehicle_loc['VehicleLongitude'], color='orange', label='Longitude')
ax.set_xlabel('Elapsed Time (s)')
ax.set_ylabel('Latitude (deg)', color='tab:blue')
ax2.set_ylabel('Longitude (deg)', color='orange')
ax.set_title('GPS Coordinates Over Time')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.savefig('gps_track.png', dpi=150, bbox_inches='tight')
plt.show()

lat_range = vehicle_loc['VehicleLatitude'].max() - vehicle_loc['VehicleLatitude'].min()
lon_range = vehicle_loc['VehicleLongitude'].max() - vehicle_loc['VehicleLongitude'].min()
approx_ns_m = lat_range * 111320
approx_ew_m = lon_range * 111320 * np.cos(np.radians(36.08))
print(f'GPS bounding box: {lat_range:.7f} deg lat x {lon_range:.7f} deg lon')
print(f'Approximate extent: {approx_ns_m:.0f}m N-S x {approx_ew_m:.0f}m E-W')

GPS bounding box: 0.0005472 deg lat x 0.0011871 deg lon
Approximate extent: 61m N-S x 107m E-W


## 4. Autonomy State and AV Light Status

The AVState message (0x011) reports whether the autonomous system is active and which control subsystems (steering, braking, propulsion) are engaged. The AVLight message (0x010) reports the blue AV indicator light status.

In [4]:
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)

autonomy_labels = {0: 'Fault', 1: 'Inactive', 2: 'Active'}
light_status_labels = {1: 'Off', 2: 'On', 3: 'Flashing'}
light_color_labels = {0: 'Off', 1: 'Blue', 2: 'Red', 3: 'Green', 4: 'White', 5: 'Amber'}

ax = axes[0]
ax.step(av_state['elapsed_s'], av_state['GlobalAutonomyStatus'], where='post', linewidth=1.5)
ax.set_ylabel('Autonomy Status')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Fault', 'Inactive', 'Active'])
ax.set_title('Autonomy Engagement Timeline')
ax.fill_between(av_state['elapsed_s'], 0, av_state['GlobalAutonomyStatus'],
                alpha=0.2, step='post')

ax = axes[1]
ax.step(av_state['elapsed_s'], av_state['SteeringCtrlActive'], where='post',
        label='Steering', linewidth=1.5)
ax.step(av_state['elapsed_s'], av_state['FrictionBrakeCtrlActive'], where='post',
        label='Brake', linewidth=1.5)
ax.step(av_state['elapsed_s'], av_state['PropulsionCtrlActive'], where='post',
        label='Propulsion', linewidth=1.5)
ax.set_ylabel('Control Active')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Off', 'On'])
ax.set_title('Control Subsystem Engagement')
ax.legend(loc='right')

ax = axes[2]
ax.step(av_light['elapsed_s'], av_light['AVLightStatus'], where='post', linewidth=1.5, color='dodgerblue')
ax.set_ylabel('Light Status')
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(['Off', 'On', 'Flashing'])
ax.set_title('AV Indicator Light Status')

ax = axes[3]
ax.step(av_light['elapsed_s'], av_light['AVLightColor'], where='post', linewidth=1.5, color='blue')
ax.set_ylabel('Light Color')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Off', 'Blue', 'Red', 'Green'])
ax.set_title('AV Indicator Light Color')
ax.set_xlabel('Elapsed Time (s)')

plt.tight_layout()
plt.savefig('autonomy_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

total_time = av_state['elapsed_s'].iloc[-1]
active_mask = av_state['GlobalAutonomyStatus'] == 2
if active_mask.any():
    active_times = av_state.loc[active_mask, 'elapsed_s']
    dt = av_state['elapsed_s'].diff().fillna(0.1)
    active_duration = dt[active_mask].sum()
else:
    active_duration = 0

print(f'Run duration: {total_time:.1f} s')
print(f'Autonomy ACTIVE duration: ~{active_duration:.1f} s ({100*active_duration/total_time:.1f}%)')
print(f'Autonomy INACTIVE duration: ~{total_time - active_duration:.1f} s')

Run duration: 196.4 s
Autonomy ACTIVE duration: ~139.8 s (71.2%)
Autonomy INACTIVE duration: ~56.6 s


## 5. Object Detection Analysis

The perception system reports detected objects via four linked messages:
- **Objects (0x020):** count of currently tracked objects
- **Object_TrackA (0x021):** type, relative position, velocity, confidence
- **Object_TrackB (0x022):** dimensions, sensor sources
- **Object_TrackC/D (0x023/0x024):** absolute GPS position, dynamic property

Object types: 0=Unknown, 1=Car, 2=Large Vehicle (semi), 3=Motorcycle/Bicycle, 4=Pedestrian, 5=Fixed Object, 6=Animal, 7=Gate/Railroad

In [5]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

ax = axes[0]
ax.step(objects_hdr['elapsed_s'], objects_hdr['Current_Object_Count'], where='post',
        linewidth=1.5, color='darkblue')
ax.fill_between(objects_hdr['elapsed_s'], 0, objects_hdr['Current_Object_Count'],
                alpha=0.2, step='post')
ax.set_ylabel('Object Count')
ax.set_title('Number of Detected Objects Over Time')
ax.set_ylim(bottom=0)

obj_type_map = {0: 'Unknown', 1: 'Car', 2: 'Large Vehicle', 3: 'Motorcycle/Bicycle',
                4: 'Pedestrian', 5: 'Fixed Object', 6: 'Animal', 7: 'Gate/Railroad'}
ax = axes[1]
obj_types = obj_track_a['ObjObjectType'].map(obj_type_map)
for otype in obj_type_map.values():
    mask = obj_types == otype
    if mask.any():
        ax.scatter(obj_track_a.loc[mask, 'elapsed_s'], obj_track_a.loc[mask, 'ObjObjectType'],
                   s=5, label=otype, alpha=0.6)
ax.set_ylabel('Object Type')
ax.set_yticks(list(obj_type_map.keys()))
ax.set_yticklabels(list(obj_type_map.values()))
ax.set_title('Detected Object Types Over Time')
ax.legend(loc='upper right', markerscale=3)

conf_map = {0: 'Invalid', 1: 'Highly Speculative', 2: 'Speculative', 3: 'Confident'}
ax = axes[2]
ax.scatter(obj_track_a['elapsed_s'], obj_track_a['Confidence'],
           c=obj_track_a['ObjObjectType'], cmap='Set1', s=5, alpha=0.5)
ax.set_ylabel('Confidence')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Invalid', 'Highly Spec.', 'Speculative', 'Confident'])
ax.set_title('Object Detection Confidence Over Time')
ax.set_xlabel('Elapsed Time (s)')

plt.tight_layout()
plt.savefig('object_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print('Object Type Distribution:')
for ot, name in obj_type_map.items():
    cnt = (obj_track_a['ObjObjectType'] == ot).sum()
    if cnt > 0:
        print(f'  {name}: {cnt} detections')

print(f'\nConfidence Distribution:')
for c, name in conf_map.items():
    cnt = (obj_track_a['Confidence'] == c).sum()
    if cnt > 0:
        print(f'  {name}: {cnt} ({100*cnt/len(obj_track_a):.1f}%)')

print(f'\nDetection Range:')
valid = obj_track_a[obj_track_a['Confidence'] > 0]
if len(valid) > 0:
    print(f'  Longitudinal: {valid["LongPos"].min():.1f}m to {valid["LongPos"].max():.1f}m')
    print(f'  Lateral: {valid["LatPos"].min():.1f}m to {valid["LatPos"].max():.1f}m')

print(f'\nSensor Sources (from Object_TrackB):')
print(f'  Camera: {(obj_track_b["Object_Source_Camera"]==1).sum()} detections')
print(f'  Lidar:  {(obj_track_b["Object_Source_Lidar"]==1).sum()} detections')
print(f'  Radar:  {(obj_track_b["Object_Source_Radar"]==1).sum()} detections')

Object Type Distribution:
  Car: 1964 detections
  Motorcycle/Bicycle: 1 detections

Confidence Distribution:
  Invalid: 231 (11.8%)
  Highly Speculative: 736 (37.5%)
  Speculative: 608 (30.9%)
  Confident: 390 (19.8%)

Detection Range:
  Longitudinal: 0.0m to 74.9m
  Lateral: -23.9m to 20.0m

Sensor Sources (from Object_TrackB):
  Camera: 1965 detections
  Lidar:  1715 detections
  Radar:  0 detections


In [6]:
fig, ax = plt.subplots(figsize=(10, 8))

valid_obj = obj_track_a[obj_track_a['Confidence'] > 0]
sc = ax.scatter(valid_obj['LatPos'], valid_obj['LongPos'],
                c=valid_obj['ObjObjectType'], cmap='Set1', s=8, alpha=0.4)
ax.set_xlabel('Lateral Position (m) [+ = right]')
ax.set_ylabel('Longitudinal Position (m) [ahead]')
ax.set_title('Object Detections in Vehicle Frame (Bird\'s Eye View)')
ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax.plot(0, 0, 'k^', markersize=15, label='Ego Vehicle')

handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.Set1(i/7),
           markersize=8, label=obj_type_map.get(i, f'Type {i}'))
           for i in sorted(valid_obj['ObjObjectType'].unique())]
handles.append(plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='black',
               markersize=10, label='Ego Vehicle'))
ax.legend(handles=handles, loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('object_birdseye.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Traffic Signal Detection Analysis

The TrafficSignalHeads (0x030) and TrafficSignalHead_TrackA (0x031) messages report detected traffic signal heads including their position, type, and which lights are illuminated. This is critical for scoring intersection behavior.

In [7]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

ax = axes[0]
ax.step(sig_hdr['elapsed_s'], sig_hdr['Current_Signal_Head_Count'], where='post',
        linewidth=1.5, color='darkgreen')
ax.set_ylabel('Signal Count')
ax.set_title('Number of Detected Traffic Signals')
ax.set_ylim(bottom=0)

ax = axes[1]
ax.step(sig_track_a['elapsed_s'], sig_track_a['IllumLtGreenBall'], where='post',
        linewidth=2, color='green', label='Green Ball')
ax.step(sig_track_a['elapsed_s'], sig_track_a['IllumLtYellowBall'], where='post',
        linewidth=2, color='goldenrod', label='Yellow Ball')
ax.step(sig_track_a['elapsed_s'], sig_track_a['IllumLtRedBall'], where='post',
        linewidth=2, color='red', label='Red Ball')
ax.set_ylabel('Light State')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Off', 'On'])
ax.set_title('Traffic Signal Light States (Ball Indicators)')
ax.legend(loc='right')

ax = axes[2]
ax.plot(sig_track_a['elapsed_s'], sig_track_a['LongPos'], linewidth=1, color='teal')
ax.set_ylabel('Distance Ahead (m)')
ax.set_title('Traffic Signal Longitudinal Distance')

ax = axes[3]
conf_colors = {0: 'red', 1: 'orange', 2: 'yellow', 3: 'green'}
ax.scatter(sig_track_a['elapsed_s'], sig_track_a['Confidence'], s=5,
           c=sig_track_a['Confidence'].map(conf_colors), alpha=0.5)
ax.set_ylabel('Confidence')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Invalid', 'Highly Spec.', 'Speculative', 'Confident'])
ax.set_title('Traffic Signal Detection Confidence')
ax.set_xlabel('Elapsed Time (s)')

plt.tight_layout()
plt.savefig('traffic_signals.png', dpi=150, bbox_inches='tight')
plt.show()

green_on = (sig_track_a['IllumLtGreenBall'] == 1).sum()
yellow_on = (sig_track_a['IllumLtYellowBall'] == 1).sum()
red_on = (sig_track_a['IllumLtRedBall'] == 1).sum()
total_sig = len(sig_track_a)
print(f'Traffic Signal Light State Distribution ({total_sig} samples):')
print(f'  Green:  {green_on} samples ({100*green_on/total_sig:.1f}%)')
print(f'  Yellow: {yellow_on} samples ({100*yellow_on/total_sig:.1f}%)')
print(f'  Red:    {red_on} samples ({100*red_on/total_sig:.1f}%)')
print(f'  None:   {total_sig - green_on - yellow_on - red_on} samples')
print(f'\nSignal distance range: {sig_track_a["LongPos"].min():.1f}m to {sig_track_a["LongPos"].max():.1f}m')

confident_sig = (sig_track_a['Confidence'] == 3).sum()
print(f'\nHigh-confidence detections: {confident_sig} ({100*confident_sig/total_sig:.1f}%)')

Traffic Signal Light State Distribution (3930 samples):
  Green:  68 samples (1.7%)
  Yellow: 888 samples (22.6%)
  Red:    0 samples (0.0%)
  None:   2974 samples

Signal distance range: 10.4m to 28.4m

High-confidence detections: 68 (1.7%)


## 7. Traffic Sign Detection Analysis

TrafficSigns (0x040) and TrafficSign_TrackA (0x041) report detected road signs.

Sign types: 0=Unknown, 1=Speed Limit (MPH), 2=Speed Limit (KPH), 3=Stop, 4=Yield, 5=No Left Turn, 6=No Right Turn, 7=Do Not Enter, 8=Left Turn Only, 9=Right Turn Only, 10=Railroad Crossing, 11=Crosswalk, 12=No Turn on Red, 13=No U-Turn, 14=One Way (Left), 15=One Way (Right)

In [8]:
sign_type_map = {
    0: 'Unknown', 1: 'Speed Limit (MPH)', 2: 'Speed Limit (KPH)', 3: 'Stop',
    4: 'Yield', 5: 'No Left Turn', 6: 'No Right Turn', 7: 'Do Not Enter',
    8: 'Left Turn Only', 9: 'Right Turn Only', 10: 'Railroad Crossing',
    11: 'Crosswalk', 12: 'No Turn on Red', 13: 'No U-Turn',
    14: 'One Way (Left)', 15: 'One Way (Right)'
}

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

ax = axes[0]
ax.step(sign_hdr['elapsed_s'], sign_hdr['Current_Sign_Count'], where='post',
        linewidth=1.5, color='purple')
ax.fill_between(sign_hdr['elapsed_s'], 0, sign_hdr['Current_Sign_Count'],
                alpha=0.2, step='post', color='purple')
ax.set_ylabel('Sign Count')
ax.set_title('Number of Detected Traffic Signs')
ax.set_ylim(bottom=0)

ax = axes[1]
valid_signs = sign_track_a[sign_track_a['Confidence'] > 0]
for st, name in sign_type_map.items():
    mask = valid_signs['Sign_Type'] == st
    if mask.any():
        ax.scatter(valid_signs.loc[mask, 'elapsed_s'], valid_signs.loc[mask, 'Sign_Type'],
                   s=10, label=name, alpha=0.6)
ax.set_ylabel('Sign Type')
ax.set_title('Detected Sign Types Over Time')
ax.legend(loc='upper right', markerscale=2, fontsize=9)

ax = axes[2]
ax.plot(sign_track_a['elapsed_s'], sign_track_a['LongPos'], '.', markersize=2, alpha=0.4)
ax.set_ylabel('Distance Ahead (m)')
ax.set_title('Traffic Sign Longitudinal Distance')
ax.set_xlabel('Elapsed Time (s)')

plt.tight_layout()
plt.savefig('traffic_signs.png', dpi=150, bbox_inches='tight')
plt.show()

print('Detected Sign Type Distribution:')
for st in sorted(sign_track_a['Sign_Type'].unique()):
    cnt = (sign_track_a['Sign_Type'] == st).sum()
    conf_cnt = ((sign_track_a['Sign_Type'] == st) & (sign_track_a['Confidence'] >= 2)).sum()
    name = sign_type_map.get(int(st), f'Type {int(st)}')
    print(f'  {name}: {cnt} total, {conf_cnt} with confidence >= speculative')

Detected Sign Type Distribution:
  Unknown: 349 total, 0 with confidence >= speculative
  Speed Limit (MPH): 204 total, 12 with confidence >= speculative
  Speed Limit (KPH): 134 total, 10 with confidence >= speculative
  Stop: 1021 total, 511 with confidence >= speculative
  Crosswalk: 58 total, 16 with confidence >= speculative
  No Turn on Red: 105 total, 31 with confidence >= speculative
  No U-Turn: 3 total, 0 with confidence >= speculative
  One Way (Right): 91 total, 88 with confidence >= speculative


## 8. Perception Quality Metrics Summary

This section summarizes the overall perception system performance across all detection categories.

In [9]:
print('=' * 70)
print('PERCEPTION SYSTEM PERFORMANCE SUMMARY')
print('=' * 70)

print(f'\n--- Run Overview ---')
print(f'Duration: {total_time:.1f} seconds')
print(f'Autonomy active: ~{active_duration:.1f}s ({100*active_duration/total_time:.1f}%)')

print(f'\n--- Object Detection ---')
print(f'Max simultaneous objects: {objects_hdr["Current_Object_Count"].max()}')
print(f'Avg simultaneous objects: {objects_hdr["Current_Object_Count"].mean():.1f}')
obj_confident = (obj_track_a['Confidence'] == 3).sum()
print(f'High-confidence detections: {obj_confident}/{len(obj_track_a)} ({100*obj_confident/len(obj_track_a):.1f}%)')
print(f'Max detection range: {obj_track_a["LongPos"].max():.1f}m')
print(f'Camera-sourced: {(obj_track_b["Object_Source_Camera"]==1).sum()}/{len(obj_track_b)}')
print(f'Lidar-sourced:  {(obj_track_b["Object_Source_Lidar"]==1).sum()}/{len(obj_track_b)}')

print(f'\n--- Traffic Signal Recognition ---')
print(f'Signals consistently tracked: {sig_hdr["Current_Signal_Head_Count"].max()}')
print(f'Green detected: {green_on} samples')
print(f'Yellow detected: {yellow_on} samples')
print(f'Signal state change captured: {"YES" if (green_on > 0 and yellow_on > 0) else "NO"}')
print(f'Max signal detection range: {sig_track_a["LongPos"].max():.1f}m')
print(f'High-confidence signal detections: {confident_sig}/{total_sig} ({100*confident_sig/total_sig:.1f}%)')

print(f'\n--- Traffic Sign Recognition ---')
print(f'Max simultaneous signs: {sign_hdr["Current_Sign_Count"].max()}')
sign_confident = (sign_track_a['Confidence'] >= 2).sum()
print(f'Confident/speculative detections: {sign_confident}/{len(sign_track_a)} ({100*sign_confident/len(sign_track_a):.1f}%)')
unique_types = sign_track_a.loc[sign_track_a['Confidence'] > 0, 'Sign_Type'].unique()
print(f'Unique sign types detected: {len(unique_types)}')
for st in sorted(unique_types):
    print(f'  - {sign_type_map.get(int(st), f"Type {int(st)}")}')

print(f'\n--- AV Indicator Light ---')
blue_on = ((av_light['AVLightColor'] == 1) & (av_light['AVLightStatus'] == 2)).sum()
print(f'Blue light ON: {blue_on}/{len(av_light)} samples ({100*blue_on/len(av_light):.1f}%)')

PERCEPTION SYSTEM PERFORMANCE SUMMARY

--- Run Overview ---
Duration: 196.4 seconds
Autonomy active: ~139.8s (71.2%)

--- Object Detection ---
Max simultaneous objects: 5
Avg simultaneous objects: 1.4
High-confidence detections: 390/1965 (19.8%)
Max detection range: 76.8m
Camera-sourced: 1965/1965
Lidar-sourced:  1715/1965

--- Traffic Signal Recognition ---
Signals consistently tracked: 1
Green detected: 68 samples
Yellow detected: 888 samples
Signal state change captured: YES
Max signal detection range: 28.4m
High-confidence signal detections: 68/3930 (1.7%)

--- Traffic Sign Recognition ---
Max simultaneous signs: 3
Confident/speculative detections: 668/1965 (34.0%)
Unique sign types detected: 6
  - Speed Limit (MPH)
  - Speed Limit (KPH)
  - Stop
  - Crosswalk
  - No Turn on Red
  - One Way (Right)

--- AV Indicator Light ---
Blue light ON: 1398/1965 samples (71.1%)


## 9. Score Estimation

Based on the CAN log analysis, we can estimate the team's scoring performance across perception categories.

### Methodology

Scoring is estimated based on:
1. **Object detection completeness** -- were objects continuously tracked with valid confidence?
2. **Traffic signal compliance** -- was the signal state correctly identified, and did the vehicle respond appropriately?
3. **Traffic sign recognition** -- were signs detected and classified correctly?
4. **Autonomy engagement** -- what percentage of the run was the vehicle in autonomous mode?
5. **Blue light compliance** -- was the blue AV indicator light active during autonomous operation?

In [10]:
print('=' * 70)
print('SCORE ESTIMATION')
print('=' * 70)

print(f'\n1. AUTONOMY ENGAGEMENT')
pct_active = 100 * active_duration / total_time
print(f'   Autonomous mode: {pct_active:.1f}% of run')
if pct_active > 80:
    print(f'   Assessment: STRONG -- high autonomy engagement throughout run')
elif pct_active > 50:
    print(f'   Assessment: MODERATE -- autonomy engaged for majority of run')
else:
    print(f'   Assessment: NEEDS IMPROVEMENT -- low autonomy engagement')

print(f'\n2. BLUE LIGHT COMPLIANCE')
pct_blue = 100 * blue_on / len(av_light)
print(f'   Blue light active: {pct_blue:.1f}% of samples')
if pct_blue > 80:
    print(f'   Assessment: COMPLIANT -- blue light consistently on during AV mode')
else:
    print(f'   Assessment: CHECK -- blue light may not align with autonomy engagement')

print(f'\n3. OBJECT DETECTION')
pct_confident_obj = 100 * obj_confident / len(obj_track_a) if len(obj_track_a) > 0 else 0
print(f'   High-confidence: {pct_confident_obj:.1f}%')
print(f'   Sensor fusion: Camera always on, Lidar contributing {100*(obj_track_b["Object_Source_Lidar"]==1).sum()/len(obj_track_b):.1f}%')
print(f'   Max range: {obj_track_a["LongPos"].max():.1f}m')
if pct_confident_obj > 60:
    print(f'   Assessment: GOOD -- majority of detections are high-confidence')
else:
    print(f'   Assessment: MODERATE -- confidence levels could be improved')

print(f'\n4. TRAFFIC SIGNAL RECOGNITION')
pct_confident_sig = 100 * confident_sig / total_sig if total_sig > 0 else 0
print(f'   High-confidence: {pct_confident_sig:.1f}%')
print(f'   State change detected: {"Green -> Yellow transition observed" if (green_on > 0 and yellow_on > 0) else "No transition"}')
if pct_confident_sig > 50:
    print(f'   Assessment: GOOD -- traffic signal reliably detected and classified')
else:
    print(f'   Assessment: MODERATE -- signal confidence could be improved')

print(f'\n5. TRAFFIC SIGN RECOGNITION')
pct_confident_sign = 100 * sign_confident / len(sign_track_a) if len(sign_track_a) > 0 else 0
print(f'   Confident detections: {pct_confident_sign:.1f}%')
print(f'   Unique types: {len(unique_types)} sign types recognized')
if len(unique_types) >= 3 and pct_confident_sign > 30:
    print(f'   Assessment: GOOD -- multiple sign types reliably detected')
else:
    print(f'   Assessment: MODERATE -- sign detection could be improved')

print(f'\n{"=" * 70}')
print(f'NOTE: Final scores depend on ground-truth comparison and event-specific')
print(f'rubrics. This estimate is based solely on the CAN log perception data.')
print(f'{"=" * 70}')

SCORE ESTIMATION

1. AUTONOMY ENGAGEMENT
   Autonomous mode: 71.2% of run
   Assessment: MODERATE -- autonomy engaged for majority of run

2. BLUE LIGHT COMPLIANCE
   Blue light active: 71.1% of samples
   Assessment: CHECK -- blue light may not align with autonomy engagement

3. OBJECT DETECTION
   High-confidence: 19.8%
   Sensor fusion: Camera always on, Lidar contributing 87.3%
   Max range: 76.8m
   Assessment: MODERATE -- confidence levels could be improved

4. TRAFFIC SIGNAL RECOGNITION
   High-confidence: 1.7%
   State change detected: Green -> Yellow transition observed
   Assessment: MODERATE -- signal confidence could be improved

5. TRAFFIC SIGN RECOGNITION
   Confident detections: 34.0%
   Unique types: 6 sign types recognized
   Assessment: GOOD -- multiple sign types reliably detected

NOTE: Final scores depend on ground-truth comparison and event-specific
rubrics. This estimate is based solely on the CAN log perception data.


## 10. Conclusion

This analysis demonstrates the team's CAN log data processing pipeline:

1. **Data Collection:** Raw CAN data was captured on HSCAN3 using the NeoVI logger in `.vsb` format
2. **Data Conversion:** The binary `.vsb` file was converted to `.csv` using the `ICS-VSBIO` Python library
3. **Signal Decoding:** Raw CAN bytes were decoded into physical signals using the `ADC2_SC_2024.2.dbc` database via `cantools`
4. **Analysis:** Decoded signals were analyzed for perception system performance across object detection, traffic signal recognition, and traffic sign recognition
5. **Visualization:** GPS track, autonomy engagement timeline, detection confidence, and signal/sign states were plotted for inspection

### Software Used
- **ICS-VSBIO** (v1.5.5) -- Intrepid Control Systems VSB file reader
- **cantools** -- CAN DBC database decoder
- **pandas / matplotlib** -- Data analysis and visualization
- **Python 3.10** on Linux (NVIDIA Jetson / aarch64)